# MaskGuard — Deteksi Penggunaan Masker Wajah dengan YOLO26n

Notebook ini membangun model **object detection** untuk mendeteksi tiga kondisi pada wajah:

1. `with_mask` — masker digunakan dengan benar.
2. `without_mask` — tidak menggunakan masker.
3. `mask_weared_incorrect` — masker digunakan secara tidak tepat.

## Permasalahan

Pemeriksaan penggunaan masker secara manual pada area padat membutuhkan petugas, memakan waktu, dan tidak konsisten. Sistem ini menawarkan deteksi otomatis berbasis kamera atau gambar agar proses pemantauan lebih cepat.

## Solusi

Model **YOLO26n** dilatih menggunakan transfer learning pada dataset *Face Mask Detection*. Alur proyek mencakup pengunduhan dataset, konversi anotasi Pascal VOC ke format YOLO, pembagian data, pelatihan, evaluasi, inferensi gambar, foto webcam, video, dan ekspor model.

> **Catatan etika:** proyek ini hanya mendeteksi status masker. Sistem tidak melakukan identifikasi atau pengenalan identitas wajah.

## 1. Persiapan lingkungan

Untuk Google Colab, aktifkan GPU melalui:

`Runtime → Change runtime type → T4 GPU`

Sel berikut menginstal dependensi utama. Setelah instalasi selesai, lanjutkan ke sel berikutnya.

In [ ]:
!pip install -q -U ultralytics kagglehub opencv-python-headless pyyaml

In [ ]:
from pathlib import Path
import os
import random
import shutil
import xml.etree.ElementTree as ET
from collections import Counter

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import yaml

from PIL import Image
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path("/content/maskguard_yolo26n")
RAW_ROOT = PROJECT_ROOT / "raw"
DATASET_ROOT = PROJECT_ROOT / "dataset"
RUNS_ROOT = PROJECT_ROOT / "runs"
DOCS_ROOT = PROJECT_ROOT / "docs"

for directory in [RAW_ROOT, DATASET_ROOT, RUNS_ROOT, DOCS_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print(f"PyTorch       : {torch.__version__}")
print(f"CUDA tersedia : {torch.cuda.is_available()}")
print(f"Device        : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")

## 2. Unduh dataset

Dataset publik diunduh menggunakan `kagglehub`. Notebook kemudian mencari folder `images` dan `annotations` secara otomatis agar tidak bergantung pada nomor versi folder Kaggle.

Apabila Kaggle meminta autentikasi, buka **Kaggle → Settings → API → Create New Token**, lalu unggah `kaggle.json` ke Colab.

In [ ]:
import kagglehub

DATASET_HANDLE = "andrewmvd/face-mask-detection"

try:
    downloaded_path = Path(kagglehub.dataset_download(DATASET_HANDLE))
except Exception as exc:
    raise RuntimeError(
        "Dataset gagal diunduh. Pastikan koneksi aktif. "
        "Jika Kaggle meminta autentikasi, unggah kaggle.json ke Colab."
    ) from exc

print("Dataset tersimpan di:", downloaded_path)

image_candidates = [
    p for p in downloaded_path.rglob("*")
    if p.is_dir() and p.name.lower() == "images"
]
annotation_candidates = [
    p for p in downloaded_path.rglob("*")
    if p.is_dir() and p.name.lower() == "annotations"
]

if not image_candidates or not annotation_candidates:
    raise FileNotFoundError(
        "Folder images/annotations tidak ditemukan. "
        "Periksa struktur dataset yang telah diunduh."
    )

RAW_IMAGES = image_candidates[0]
RAW_ANNOTATIONS = annotation_candidates[0]

print("Folder gambar   :", RAW_IMAGES)
print("Folder anotasi  :", RAW_ANNOTATIONS)

## 3. Audit dataset

Audit awal memastikan:

- jumlah gambar dan anotasi masuk akal;
- nama kelas terbaca;
- pasangan gambar–XML tersedia;
- distribusi objek per kelas dapat diperiksa sebelum pelatihan.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

raw_images = sorted([
    p for p in RAW_IMAGES.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
])
raw_xml = sorted(RAW_ANNOTATIONS.glob("*.xml"))

if not raw_images:
    raise ValueError("Tidak ada gambar yang ditemukan.")
if not raw_xml:
    raise ValueError("Tidak ada anotasi XML yang ditemukan.")

class_counter = Counter()
invalid_xml = []

for xml_path in raw_xml:
    try:
        root = ET.parse(xml_path).getroot()
        for obj in root.findall("object"):
            name_node = obj.find("name")
            if name_node is not None and name_node.text:
                class_counter[name_node.text.strip()] += 1
    except ET.ParseError:
        invalid_xml.append(xml_path.name)

print(f"Jumlah gambar       : {len(raw_images)}")
print(f"Jumlah file XML     : {len(raw_xml)}")
print(f"XML tidak valid     : {len(invalid_xml)}")
print("Distribusi objek    :")
for label, count in class_counter.most_common():
    print(f"  - {label}: {count}")

if invalid_xml:
    print("Contoh XML bermasalah:", invalid_xml[:5])

## 4. Konversi anotasi Pascal VOC menjadi format YOLO

Format bounding box YOLO:

`class_id x_center y_center width height`

Semua koordinat dinormalisasi ke rentang 0–1. Dataset dibagi menjadi:

- 70% data latih;
- 20% data validasi;
- 10% data uji.

In [ ]:
CLASS_NAMES = [
    "with_mask",
    "without_mask",
    "mask_weared_incorrect",
]
CLASS_TO_ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}

SPLIT_RATIOS = {
    "train": 0.70,
    "val": 0.20,
    "test": 0.10,
}

def find_image_for_stem(stem: str) -> Path | None:
    for ext in IMAGE_EXTENSIONS:
        candidate = RAW_IMAGES / f"{stem}{ext}"
        if candidate.exists():
            return candidate
        candidate_upper = RAW_IMAGES / f"{stem}{ext.upper()}"
        if candidate_upper.exists():
            return candidate_upper
    return None

pairs = []
skipped = []

for xml_path in raw_xml:
    image_path = find_image_for_stem(xml_path.stem)
    if image_path is None:
        skipped.append(xml_path.name)
    else:
        pairs.append((image_path, xml_path))

if len(pairs) < 10:
    raise ValueError("Pasangan gambar dan anotasi terlalu sedikit untuk dibagi.")

random.Random(SEED).shuffle(pairs)

n_total = len(pairs)
n_train = int(n_total * SPLIT_RATIOS["train"])
n_val = int(n_total * SPLIT_RATIOS["val"])

split_pairs = {
    "train": pairs[:n_train],
    "val": pairs[n_train:n_train + n_val],
    "test": pairs[n_train + n_val:],
}

print("Pasangan valid :", len(pairs))
print("Tanpa gambar   :", len(skipped))
for split, items in split_pairs.items():
    print(f"{split:>5}: {len(items)}")

In [ ]:
def voc_box_to_yolo(
    xmin: float,
    ymin: float,
    xmax: float,
    ymax: float,
    image_width: int,
    image_height: int,
):
    xmin = max(0.0, min(float(xmin), image_width))
    xmax = max(0.0, min(float(xmax), image_width))
    ymin = max(0.0, min(float(ymin), image_height))
    ymax = max(0.0, min(float(ymax), image_height))

    box_width = max(0.0, xmax - xmin)
    box_height = max(0.0, ymax - ymin)

    x_center = xmin + box_width / 2.0
    y_center = ymin + box_height / 2.0

    return (
        x_center / image_width,
        y_center / image_height,
        box_width / image_width,
        box_height / image_height,
    )

def convert_xml_to_yolo(xml_path: Path) -> list[str]:
    root = ET.parse(xml_path).getroot()

    size = root.find("size")
    if size is None:
        raise ValueError(f"Elemen size tidak ditemukan pada {xml_path.name}")

    image_width = int(float(size.findtext("width", "0")))
    image_height = int(float(size.findtext("height", "0")))

    if image_width <= 0 or image_height <= 0:
        raise ValueError(f"Ukuran gambar tidak valid pada {xml_path.name}")

    yolo_lines = []

    for obj in root.findall("object"):
        class_name = (obj.findtext("name") or "").strip()
        if class_name not in CLASS_TO_ID:
            continue

        bndbox = obj.find("bndbox")
        if bndbox is None:
            continue

        xmin = float(bndbox.findtext("xmin", "0"))
        ymin = float(bndbox.findtext("ymin", "0"))
        xmax = float(bndbox.findtext("xmax", "0"))
        ymax = float(bndbox.findtext("ymax", "0"))

        x_center, y_center, box_width, box_height = voc_box_to_yolo(
            xmin, ymin, xmax, ymax, image_width, image_height
        )

        if box_width <= 0 or box_height <= 0:
            continue

        yolo_lines.append(
            f"{CLASS_TO_ID[class_name]} "
            f"{x_center:.6f} {y_center:.6f} "
            f"{box_width:.6f} {box_height:.6f}"
        )

    return yolo_lines

# Bersihkan hasil konversi lama agar rerun tetap konsisten.
for folder_name in ["images", "labels"]:
    target = DATASET_ROOT / folder_name
    if target.exists():
        shutil.rmtree(target)

conversion_errors = []

for split, items in split_pairs.items():
    image_out = DATASET_ROOT / "images" / split
    label_out = DATASET_ROOT / "labels" / split
    image_out.mkdir(parents=True, exist_ok=True)
    label_out.mkdir(parents=True, exist_ok=True)

    for image_path, xml_path in items:
        try:
            yolo_lines = convert_xml_to_yolo(xml_path)
            shutil.copy2(image_path, image_out / image_path.name)
            (label_out / f"{image_path.stem}.txt").write_text(
                "\n".join(yolo_lines),
                encoding="utf-8",
            )
        except Exception as exc:
            conversion_errors.append((xml_path.name, str(exc)))

print("Konversi selesai.")
for split in split_pairs:
    image_count = len(list((DATASET_ROOT / "images" / split).glob("*")))
    label_count = len(list((DATASET_ROOT / "labels" / split).glob("*.txt")))
    print(f"{split:>5}: {image_count} gambar, {label_count} label")

print("Error konversi:", len(conversion_errors))
if conversion_errors:
    print("Contoh:", conversion_errors[:3])

## 5. Buat konfigurasi `data.yaml`

In [ ]:
data_config = {
    "path": str(DATASET_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {idx: name for idx, name in enumerate(CLASS_NAMES)},
}

DATA_YAML = DATASET_ROOT / "data.yaml"
DATA_YAML.write_text(
    yaml.safe_dump(data_config, sort_keys=False),
    encoding="utf-8",
)

print(DATA_YAML.read_text(encoding="utf-8"))

## 6. Verifikasi visual anotasi

Tahap ini penting untuk mendeteksi kesalahan koordinat sebelum training. Kotak harus tepat mengelilingi wajah dan label kelas harus sesuai.

In [ ]:
def load_yolo_labels(label_path: Path):
    boxes = []
    if not label_path.exists():
        return boxes

    for line in label_path.read_text(encoding="utf-8").splitlines():
        parts = line.split()
        if len(parts) != 5:
            continue
        class_id = int(parts[0])
        x_center, y_center, width, height = map(float, parts[1:])
        boxes.append((class_id, x_center, y_center, width, height))
    return boxes

def draw_yolo_annotations(image_path: Path, label_path: Path):
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        raise ValueError(f"Gambar tidak dapat dibaca: {image_path}")

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    height, width = image_rgb.shape[:2]

    for class_id, x_center, y_center, box_width, box_height in load_yolo_labels(label_path):
        x1 = int((x_center - box_width / 2) * width)
        y1 = int((y_center - box_height / 2) * height)
        x2 = int((x_center + box_width / 2) * width)
        y2 = int((y_center + box_height / 2) * height)

        cv2.rectangle(image_rgb, (x1, y1), (x2, y2), 2)
        cv2.putText(
            image_rgb,
            CLASS_NAMES[class_id],
            (x1, max(20, y1 - 7)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            2,
        )

    return image_rgb

train_images = sorted((DATASET_ROOT / "images" / "train").glob("*"))
sample_images = random.Random(SEED).sample(train_images, k=min(4, len(train_images)))

plt.figure(figsize=(14, 10))
for index, image_path in enumerate(sample_images, start=1):
    label_path = DATASET_ROOT / "labels" / "train" / f"{image_path.stem}.txt"
    rendered = draw_yolo_annotations(image_path, label_path)
    plt.subplot(2, 2, index)
    plt.imshow(rendered)
    plt.title(image_path.name)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 7. Training YOLO26n

`yolo26n.pt` dipilih karena varian nano relatif ringan dan cocok untuk eksperimen Colab serta inferensi real-time.

Parameter utama dapat diubah pada bagian konfigurasi. Untuk hasil tugas yang lebih stabil, gunakan sekitar 30–50 epoch. Gunakan `FAST_MODE=True` hanya untuk uji alur program.

In [ ]:
FAST_MODE = False

EPOCHS = 3 if FAST_MODE else 30
IMAGE_SIZE = 640
BATCH_SIZE = 8 if DEVICE == "cpu" else 16
RUN_NAME = "maskguard_yolo26n"

model = YOLO("yolo26n.pt")

train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=2,
    patience=10,
    pretrained=True,
    seed=SEED,
    deterministic=True,
    project=str(RUNS_ROOT),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    verbose=True,
)

BEST_WEIGHTS = RUNS_ROOT / RUN_NAME / "weights" / "best.pt"
LAST_WEIGHTS = RUNS_ROOT / RUN_NAME / "weights" / "last.pt"

if not BEST_WEIGHTS.exists():
    raise FileNotFoundError(f"Bobot best.pt tidak ditemukan: {BEST_WEIGHTS}")

print("Best weights:", BEST_WEIGHTS)

## 8. Evaluasi model pada data uji

Metrik yang ditampilkan:

- **Precision**: proporsi deteksi positif yang benar.
- **Recall**: proporsi objek sebenarnya yang berhasil ditemukan.
- **mAP50**: mean Average Precision pada IoU 0,50.
- **mAP50–95**: rata-rata AP pada IoU 0,50 sampai 0,95.

In [ ]:
best_model = YOLO(str(BEST_WEIGHTS))

metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    project=str(RUNS_ROOT),
    name=f"{RUN_NAME}_test",
    exist_ok=True,
    plots=True,
)

summary = {
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "mAP50": float(metrics.box.map50),
    "mAP50-95": float(metrics.box.map),
}

print("Ringkasan evaluasi")
for key, value in summary.items():
    print(f"{key:>10}: {value:.4f}")

In [ ]:
evaluation_dir = RUNS_ROOT / f"{RUN_NAME}_test"
plot_names = [
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "F1_curve.png",
]

for plot_name in plot_names:
    plot_path = evaluation_dir / plot_name
    if plot_path.exists():
        display(Image.open(plot_path))
    else:
        print(f"Belum ditemukan: {plot_path}")

## 9. Inferensi pada gambar uji

Hasil prediksi disimpan ke folder `runs/maskguard_predict/`.

In [ ]:
test_images = sorted((DATASET_ROOT / "images" / "test").glob("*"))
if not test_images:
    raise ValueError("Data uji kosong.")

sample_test_image = test_images[0]

prediction_results = best_model.predict(
    source=str(sample_test_image),
    conf=0.25,
    imgsz=IMAGE_SIZE,
    device=DEVICE,
    save=True,
    project=str(RUNS_ROOT),
    name="maskguard_predict",
    exist_ok=True,
)

annotated_bgr = prediction_results[0].plot()
annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 8))
plt.imshow(annotated_rgb)
plt.title("Hasil Deteksi Masker")
plt.axis("off")
plt.show()

# Simpan satu gambar untuk dokumentasi GitHub.
documentation_image = DOCS_ROOT / "hasil_deteksi.jpg"
Image.fromarray(annotated_rgb).save(documentation_image)
print("Dokumentasi hasil:", documentation_image)

## 10. Prediksi gambar yang diunggah pengguna

Jalankan sel ini di Google Colab, lalu pilih gambar `.jpg`, `.jpeg`, atau `.png`.

In [ ]:
try:
    from google.colab import files
except ImportError as exc:
    raise RuntimeError("Fitur upload ini hanya tersedia di Google Colab.") from exc

uploaded = files.upload()

for filename, file_bytes in uploaded.items():
    suffix = Path(filename).suffix.lower()
    if suffix not in IMAGE_EXTENSIONS:
        print(f"Dilewati karena format tidak didukung: {filename}")
        continue

    upload_path = PROJECT_ROOT / Path(filename).name
    upload_path.write_bytes(file_bytes)

    results = best_model.predict(
        source=str(upload_path),
        conf=0.25,
        imgsz=IMAGE_SIZE,
        device=DEVICE,
    )

    plotted_bgr = results[0].plot()
    plotted_rgb = cv2.cvtColor(plotted_bgr, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(12, 8))
    plt.imshow(plotted_rgb)
    plt.title(f"Hasil: {filename}")
    plt.axis("off")
    plt.show()

## 11. Foto webcam di Google Colab

Implementasi berikut mengambil **satu foto per eksekusi**. Pendekatan ini lebih stabil daripada loop yang meminta akses kamera berulang kali.

In [ ]:
from base64 import b64decode
from IPython.display import Javascript, display
from google.colab.output import eval_js

def take_webcam_photo(filename="webcam_capture.jpg", quality=0.85):
    javascript = Javascript(r'''
        async function capturePhoto(quality) {
            const container = document.createElement('div');
            const video = document.createElement('video');
            const captureButton = document.createElement('button');
            const cancelButton = document.createElement('button');

            captureButton.textContent = 'Ambil Foto';
            cancelButton.textContent = 'Batal';

            container.appendChild(video);
            container.appendChild(document.createElement('br'));
            container.appendChild(captureButton);
            container.appendChild(cancelButton);
            document.body.appendChild(container);

            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            video.srcObject = stream;
            await video.play();

            const result = await new Promise((resolve) => {
                captureButton.onclick = () => resolve('capture');
                cancelButton.onclick = () => resolve('cancel');
            });

            if (result === 'cancel') {
                stream.getTracks().forEach(track => track.stop());
                container.remove();
                return null;
            }

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);

            stream.getTracks().forEach(track => track.stop());
            container.remove();

            return canvas.toDataURL('image/jpeg', quality);
        }
    ''')

    display(javascript)
    data = eval_js(f"capturePhoto({quality})")

    if data is None:
        print("Pengambilan foto dibatalkan.")
        return None

    binary = b64decode(data.split(",")[1])
    output_path = PROJECT_ROOT / filename
    output_path.write_bytes(binary)
    return output_path

webcam_image = take_webcam_photo()

if webcam_image is not None:
    webcam_results = best_model.predict(
        source=str(webcam_image),
        conf=0.25,
        imgsz=IMAGE_SIZE,
        device=DEVICE,
    )

    webcam_bgr = webcam_results[0].plot()
    webcam_rgb = cv2.cvtColor(webcam_bgr, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(12, 8))
    plt.imshow(webcam_rgb)
    plt.title("Deteksi dari Webcam")
    plt.axis("off")
    plt.show()

## 12. Ekspor model

Format ONNX berguna untuk integrasi ke aplikasi desktop, web backend, atau perangkat edge.

In [ ]:
exported_path = best_model.export(
    format="onnx",
    imgsz=IMAGE_SIZE,
    simplify=True,
)

print("Model ONNX:", exported_path)

## 13. Unduh artefak hasil training

File ZIP mencakup bobot `best.pt`, kurva evaluasi, hasil prediksi, konfigurasi dataset, dan dokumentasi gambar.

In [ ]:
import zipfile

artifact_zip = PROJECT_ROOT / "maskguard_training_artifacts.zip"

with zipfile.ZipFile(artifact_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    important_paths = [
        BEST_WEIGHTS,
        LAST_WEIGHTS,
        DATA_YAML,
        DOCS_ROOT / "hasil_deteksi.jpg",
    ]

    for path in important_paths:
        if path.exists():
            archive.write(path, arcname=path.relative_to(PROJECT_ROOT))

    for folder in [
        RUNS_ROOT / RUN_NAME,
        RUNS_ROOT / f"{RUN_NAME}_test",
        RUNS_ROOT / "maskguard_predict",
    ]:
        if folder.exists():
            for file_path in folder.rglob("*"):
                if file_path.is_file():
                    archive.write(
                        file_path,
                        arcname=file_path.relative_to(PROJECT_ROOT),
                    )

print("ZIP berhasil dibuat:", artifact_zip)

from google.colab import files
files.download(str(artifact_zip))

## Kesimpulan

Model yang dibangun menyelesaikan permasalahan pemantauan penggunaan masker dengan cara:

1. menerima gambar, foto webcam, atau video;
2. mendeteksi lokasi wajah;
3. mengklasifikasikan status masker;
4. menampilkan bounding box, label, dan confidence;
5. menyimpan hasil prediksi dan model hasil training.

Kualitas model tetap dipengaruhi oleh variasi pencahayaan, sudut wajah, ukuran objek, occlusion, kualitas kamera, serta distribusi dataset. Model sebaiknya diuji kembali pada lingkungan implementasi sebenarnya sebelum digunakan.